In [6]:
import os
import pandas as pd
# import polars as pl
import torch
from torch.utils.data import DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# from data_handling import create_few_shot_sst2_messages, SST2Dataset
from tqdm import tqdm
import argparse
import json
import warnings
from accelerate.test_utils.testing import get_backend


# def load_sst2_data():
#     splits = {
#         "train": "data/train-00000-of-00001.parquet",
#         "validation": "data/validation-00000-of-00001.parquet",
#         "test": "data/test-00000-of-00001.parquet",
#     }
#     sst2 = pl.read_parquet("hf://datasets/stanfordnlp/sst2/" + splits["train"])
#     sst2_train, sst2_test = train_test_split(sst2, test_size=0.5, random_state=42, shuffle=True, stratify=sst2["label"])
#     # sst2_test = pl.read_parquet("hf://datasets/stanfordnlp/sst2/" + splits["test"])
#     return sst2_train, sst2_test


def load_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left", add_prefix_space=False)
    tokenizer.pad_token_id = tokenizer.eos_token_id
    if "gemma" in model_id:
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=quantization_config,
        )
        # update chat template to allow for assistant role to go first
        tokenizer.chat_template = "{{ bos_token }}{% for message in messages %}\
            {% if (message['role'] == 'assistant') %}{% set role = 'model' %}\
            {% else %}{% set role = message['role'] %}{% endif %}\
            {{ '<start_of_turn>' + role + '\n' + message['content'] | trim + '<end_of_turn>\n' }}{% endfor %}\
            {% if add_generation_prompt %}{{'<start_of_turn>model\n'}}{% endif %}"
    elif "llama" in model_id:
        quantization_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", quantization_config=quantization_config, torch_dtype="auto"
        )
    elif "Falcon" in model_id:
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", quantization_config=quantization_config, torch_dtype="auto"
        )
    elif "Mistral" in model_id:
        quantization_config = BitsAndBytesConfig(load_in_4bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", quantization_config=quantization_config, torch_dtype="auto"
        )

    else:
        warnings.warn(f"Model {model_id} not supported")
        model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
    return tokenizer, model


def update_generation_config(model, tokenizer):
    generation_config_updates = {
        "pad_token_id": tokenizer.eos_token_id,
        "return_dict_in_generate": True,
        "output_scores": True,
        "do_sample": False,
        "temperature": None,
        "top_p": None,
    }
    model.generation_config.__dict__.update(generation_config_updates)
    print(model.generation_config)


def check_environment():
    assert "HF_HOME" in os.environ, "HF_HOME is not set"
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        raise ValueError("No CUDA or MPS device available")


# def prepare_data(sst2_train, sst2_test, config):
#     context_samples = sst2_train.sample(config["n_context"], seed=config["seed"])
#     # print the proportion of positive and negative samples
#     # print(context_samples["label"].value_counts(normalize=True))
#     target_samples = sst2_test.sample(config["n_target"], seed=config["seed"])
#     target_messages = create_few_shot_sst2_messages(context_samples, target_samples)
#     return context_samples, target_samples, target_messages


# def build_dataloader(target_messages, batch_size):
#     dataset = SST2Dataset(target_messages)
#     return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=lambda x: x)


# def generate_predictions(loader, tokenizer, model, device):
#     test_preds = []
#     test_probs = []
#     for batch_messages in tqdm(loader):
#         batch_messages_str = tokenizer.apply_chat_template(
#             batch_messages,
#             tokenize=False,
#             add_generation_prompt=True,
#         )
#         batch_tokens = tokenizer.batch_encode_plus(batch_messages_str, return_tensors="pt", padding=True).to(device)
#         outputs = model.generate(  # max_new_tokens was 10 initially
#             **batch_tokens, max_new_tokens=1, output_scores=True, output_logits=True, renormalize_logits=True
#         )
#         # print(batch_messages_str)
#         # print(outputs)
#         # -3: includes the end-of-turn and end-of-sequence tokens
#         preds = tokenizer.batch_decode(outputs["sequences"][:, -1:], skip_special_tokens=True)
#         # if len(preds[0]) == 0:
#         #     print(tokenizer.batch_decode(outputs["sequences"], skip_special_tokens=False))
#         print(preds)
#         # filter out predictions that are not 0 or 1, if it can be converted to an int
#         # get indices of preds that can be converted to an int
#         # valid_indices = [i for i, pred in enumerate(preds) if pred in ["0", "1"]]
#         # convert preds to int, otherwise fill with nan
#         preds = [int(pred) if pred in ["0", "1"] else np.nan for pred in preds]
#         test_preds.extend(preds)
#         probs = torch.softmax(outputs["logits"][0], dim=-1).max(dim=-1).values.cpu().numpy().tolist()
#         test_probs.extend(probs)
#     return np.array(test_preds), np.array(test_probs)


# def save_results(test_preds, test_probs, target_samples, context_samples, config):
#     test_labels = target_samples["label"].to_numpy()
#     test_df = pd.DataFrame(
#         {
#             "target_idx": target_samples["idx"].to_numpy(),
#             "sentence": target_samples["sentence"].to_numpy(),
#             "pred": test_preds,
#             "prob": test_probs,
#             "true_label": test_labels,
#             "context_idx": [context_samples["idx"].to_list()] * config["n_target"],
#         }
#     )
#     test_df.attrs["config"] = config
#     test_df.to_csv(
#         f"results/sst2_context_{config['n_context']}_{config['model_id'].replace('/', '_')}.csv", index=False
#     )
#     test_df.to_pickle(f"results/sst2_context_{config['n_context']}_{config['model_id'].replace('/', '_')}.pkl")


# def parse_args():
#     parser = argparse.ArgumentParser(description="Run SST-2 classification.")
#     parser.add_argument("--config_file", type=str, help="Path to the JSON config file.")
#     parser.add_argument("--batch_size", type=int, default=8, help="Batch size for data loading.")
#     parser.add_argument("--seed", type=int, default=42, help="Random seed.")
#     parser.add_argument("--n_context", type=int, default=100, help="Number of context samples.")
#     parser.add_argument("--n_target", type=int, default=500, help="Number of target samples.")
#     parser.add_argument("--model_id", type=str, default="meta-llama/Llama-3.3-1B-Instruct", help="Model ID.")
#     return parser.parse_args()


# def load_config(args):
#     if args.config_file:
#         with open(args.config_file, "r") as f:
#             config = json.load(f)
#     else:
#         config = {
#             "batch_size": args.batch_size,
#             "seed": args.seed,
#             "n_context": args.n_context,
#             "n_target": args.n_target,
#             "model_id": args.model_id,
#         }
#     return config


# def main(config):
#     print(f"Running with config: {config}")
#     sst2_train, sst2_test = load_sst2_data()
#     tokenizer, model = load_model(config["model_id"])
#     update_generation_config(model, tokenizer)
#     device = check_environment()
#     # every test sample gets the same context
#     context_samples, target_samples, target_messages = prepare_data(sst2_train, sst2_test, config)
#     loader = build_dataloader(target_messages, config["batch_size"])
#     test_preds, test_probs = generate_predictions(loader, tokenizer, model, device)
#     save_results(test_preds, test_probs, target_samples, context_samples, config)

from datasets import load_dataset, Dataset, DatasetDict
from torch.utils.data import DataLoader
from transformers import PreTrainedTokenizerBase


def load_snli(test_size: int = 2500, pretrain_size: int = 2500, train_size: int = 10000, seed: int = 42) -> DatasetDict:
    ds = load_dataset("stanfordnlp/snli")
    ds = ds.filter(lambda x: x["label"] != -1)

    # Split the "train" dataset into "pretrain" and "train"
    train_test_split = ds["train"].train_test_split(test_size=pretrain_size, seed=seed)
    pretrain_dataset = train_test_split["test"]  # This will be your "pretrain" sample
    train_dataset = train_test_split["train"].select(range(train_size))  # This will be the remainder "train"

    # Create a new DatasetDict with the updated splits
    ds = DatasetDict(
        {
            "pretrain": pretrain_dataset,
            "train": train_dataset,
            "validation": ds["validation"],
            "test": ds["test"].select(range(test_size)) if test_size < len(ds["test"]) else ds["test"],
        }
    )
    return ds


def tokenize_dataset(ds: Dataset, tokenizer: PreTrainedTokenizerBase, max_length: int = 128):
    task_to_keys = {
        "snli": ("premise", "hypothesis"),
    }
    sentence1_key, sentence2_key = task_to_keys["snli"]
    # annotate examples so the model prompt is formatted correctly
    ds = ds.map(
        lambda x: {
            sentence1_key: f"{sentence1_key}: {x[sentence1_key]}",
            sentence2_key: f"{sentence2_key}: {x[sentence2_key]}",
        }
    )

    def tokenize_function(examples: Dataset, max_length: int = 128):
        """
        Tokenize the dataset.

        """
        if sentence2_key is None:
            return tokenizer(examples[sentence1_key], truncation=True)
        return tokenizer(
            examples[sentence1_key],
            examples[sentence2_key],
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        # return tokenizer(f"{sentence1_key}: {examples[sentence1_key]}", truncation=True)

    tokenized_datasets = ds.map(tokenize_function, batched=True)
    tokenized_datasets = tokenized_datasets.remove_columns(["premise", "hypothesis"])
    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    return tokenized_datasets


In [ ]:
datasets = load_snli(
        test_size=args.test_size,
        pretrain_size=args.pretrain_size,
        train_size=args.train_size,
        seed=args.seed,
    )
model_id = "meta-llama/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, num_labels=3, device_map="auto", torch_dtype="auto"
)
model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding = True
tokenizer.padding_side = "left"
model.config.pad_token_id = tokenizer.eos_token_id

tokenized_datasets = tokenize_dataset(datasets, tokenizer)

In [ ]:
# Move model to evaluation mode
model.eval()
# Use a DataLoader to iterate over the dataset
dataloader = DataLoader(tokenized_datasets["train"], batch_size=16, shuffle=False)

with torch.no_grad():
    activations_by_layer = []
    predictions = []
    for batch in dataloader:
        # print(batch)
        # construct input ids and attention mask with shape (batch_size, seq_len)
        input_ids = torch.stack(batch["input_ids"], dim=1).to(model.device)
        attention_mask = torch.stack(batch["attention_mask"], dim=1).to(
            model.device
        )
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        # first layer is the embedding layer, so we skip it
        # stack layers, compute mean across sequence length
        activations_by_layer.append(
            torch.stack(
                [
                    x.detach().cpu().to(torch.float32).mean(dim=1)
                    for x in outputs.hidden_states[1:]
                ]
            )
        )
        # get predictions
        predictions.append(outputs.logits.detach().cpu())
    pred_labels = torch.argmax(torch.cat(predictions), dim=1)
    # concatenate batches to get a single tensor of shape (n_layers, n_samples, n_features)
    activations_by_layer = torch.cat(activations_by_layer, dim=1)
    print(
        "activations_by_layer shape (n_layers, n_samples, n_features):\n",
        activations_by_layer.shape,
    )

DatasetDict({
    pretrain: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 9842
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1000
    })
})